# Register KBaseFBA.GenomeDataLakeTables Type

This notebook registers the new `KBaseFBA.GenomeDataLakeTables` type in the KBase
workspace type system using the KIDL spec from `data/KBaseFBA.spec`.

**New types being registered:**
- `KBaseFBA.GenomeDataLakeTables` — Holds DataLake tables with contextual data for a genome set
- `KBaseFBA.PangenomeData` — Sub-structure for pangenome SQLite table references

**Workflow:**
1. Load and inspect the KIDL spec
2. Check current KBaseFBA module info and ownership
3. Dry-run typespec registration (compile without saving)
4. Register the typespec for real
5. Release the module to make types available
6. Verify registration by fetching type info

**Prerequisites:**
- Module ownership of `KBaseFBA` (request via `request_module_ownership` if needed)
- Valid KBase authentication token with admin permissions
- The KIDL spec file at `notebooks/data/KBaseFBA.spec`

## Step 1: Load and Inspect the KIDL Spec

Read the KBaseFBA.spec file and display the GenomeDataLakeTables type definition
to verify the spec is correct before attempting registration.

- Input: `data/KBaseFBA.spec`
- Output: `spec_text` (full spec), `new_types` (types to register)

In [8]:
%run util.py
import os

# Load the KIDL spec file
spec_path = os.path.join(util.notebook_folder, 'data', 'KBaseFBA.spec')
print(f"Spec file: {spec_path}")
print(f"File exists: {os.path.exists(spec_path)}")

with open(spec_path, 'r') as f:
    spec_text = f.read()

print(f"Spec length: {len(spec_text):,} characters")
print(f"Spec lines: {len(spec_text.splitlines()):,}")

# Display the new type definitions (GenomeDataLakeTables and its dependencies)
print("\n" + "=" * 80)
print("NEW TYPE DEFINITIONS:")
print("=" * 80)

# Extract and display the relevant type definitions
for marker in ['GenomeSet_ref', 'table_handle_ref', 'PangenomeData', 'GenomeDataLakeTables']:
    for i, line in enumerate(spec_text.splitlines()):
        if marker in line:
            # Print surrounding context
            start = max(0, i - 2)
            end = min(len(spec_text.splitlines()), i + 15)
            context = spec_text.splitlines()[start:end]
            print(f"\n--- {marker} (line {i+1}) ---")
            for ctx_line in context:
                print(f"  {ctx_line}")
            break

# Define the new types to register
new_types = ["GenomeDataLakeTables"]
print(f"\nNew types to register: {new_types}")

# Save for later cells
util.save('spec_text', spec_text)
util.save('new_types', new_types)
util.save('registration_config', {
    'module_name': 'KBaseFBA',
    'spec_path': spec_path,
    'new_types': new_types,
    'kb_version': 'appdev'
})

2026-02-20 09:41:04,710 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-20 09:41:04,711 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-20 09:41:04,712 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-20 09:41:04,712 - __main__.NotebookUtil - INFO - Notebook name: RegisterGenomeDataLakeTables
2026-02-20 09:41:04,713 - __main__.NotebookUtil - INFO - Notebook environment detected


Spec file: /Users/chenry/Dropbox/Projects/KBDatalakeApps/notebooks/data/KBaseFBA.spec
File exists: True
Spec length: 57,415 characters
Spec lines: 2,047

NEW TYPE DEFINITIONS:

--- GenomeSet_ref (line 2014) ---
      @id ws KBaseSearch.GenomeSet
      */
      typedef string GenomeSet_ref;
  
      /*
      Reference to a handle to the tables in SQLLite format
      @id handle
      */
      typedef string table_handle_ref;
  
      /*
      Data structure for storing pangenome data
      */
      typedef structure {
          string pangenome_id;
          string pangenome_taxonomy;
          list<genome_ref> user_genomes;

--- table_handle_ref (line 2020) ---
      @id handle
      */
      typedef string table_handle_ref;
  
      /*
      Data structure for storing pangenome data
      */
      typedef structure {
          string pangenome_id;
          string pangenome_taxonomy;
          list<genome_ref> user_genomes;
          list<string> datalake_genomes;
          table_hand

## Step 2: Check KBaseFBA Module Info and Ownership

Before registering, check the current state of the KBaseFBA module:
- Module versions and ownership
- Currently registered types
- Whether we have permissions to register new types

If ownership is needed, use `request_module_ownership` (requires admin approval).

In [3]:
%run util.py

config = util.load('registration_config')
module_name = config['module_name']
kb_version = config['kb_version']

# Create workspace client for the target environment
ws = KBWSUtils(kb_version=kb_version)

# Get module info
print(f"Checking module: {module_name} on {kb_version}")
print("=" * 80)

module_info = ws.get_module_info(module_name)
if 'error' in module_info:
    print(f"Error: {module_info['error']}")
else:
    print(f"Module: {module_name}")
    print(f"Description: {module_info.get('description', 'N/A')}")
    print(f"Owners: {module_info.get('owners', [])}")
    
    # Show registered types
    types = module_info.get('types', {})
    print(f"\nRegistered types ({len(types)}):")
    for type_name, type_ver in sorted(types.items()):
        print(f"  {type_name}: {type_ver}")

# Check module versions
print("\n" + "-" * 80)
print("Module versions:")
versions = ws.list_module_versions(module_name)
if 'error' in versions:
    print(f"Error: {versions['error']}")
else:
    print(f"  Versions: {versions}")

util.save('module_info', module_info)
util.save('module_versions', versions)

2026-02-20 09:38:37,862 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-20 09:38:37,862 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-20 09:38:37,864 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-20 09:38:37,864 - __main__.NotebookUtil - INFO - Notebook name: RegisterGenomeDataLakeTables
2026-02-20 09:38:37,865 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-20 09:38:37,868 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-20 09:38:37,868 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-20 09:38:37,869 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token


Checking module: KBaseFBA on prod
Module: KBaseFBA
Description: @author chenry
Owners: ['chenry']

Registered types (22):
  KBaseFBA.Classifier-1.1: {
  "id" : "Classifier",
  "description" : "",
  "type" : "object",
  "original-type" : "kidl-structure",
  "properties" : {
    "id" : {
      "type" : "string",
      "original-type" : "kidl-string"
    },
    "attribute_type" : {
      "type" : "string",
      "original-type" : "kidl-string"
    },
    "classifier_type" : {
      "type" : "string",
      "original-type" : "kidl-string"
    },
    "trainingset_ref" : {
      "type" : "string",
      "original-type" : "kidl-string",
      "id-reference" : {
        "attributes" : [ "KBaseFBA.ClassifierTrainingSet" ],
        "id-type" : "ws"
      }
    },
    "data" : {
      "type" : "string",
      "original-type" : "kidl-string"
    },
    "readable" : {
      "type" : "string",
      "original-type" : "kidl-string"
    },
    "correctly_classified_instances" : {
      "type" : "integ

## Step 2b: Request Module Ownership (If Needed)

If you don't already own the KBaseFBA module, request ownership here.
This requires admin approval before you can register types.

**Only run this cell if Step 2 shows you are NOT an owner.**

In [ ]:
%run util.py

config = util.load('registration_config')
module_name = config['module_name']
kb_version = config['kb_version']

# Safety check - set to True to request ownership
CONFIRM_REQUEST = False  # Change to True to proceed

ws = KBWSUtils(kb_version=kb_version)

if CONFIRM_REQUEST:
    print(f"Requesting ownership of module: {module_name}")
    result = ws.request_module_ownership(module_name)
    print(f"Result: {result}")
    print("\nNote: Admin approval is required before you can register types.")
else:
    print(f"Ownership request skipped. Set CONFIRM_REQUEST = True to proceed.")
    print(f"This would request ownership of: {module_name}")

## Step 3: Dry-Run Typespec Registration

Test the typespec compilation without actually saving it.
This validates:
- KIDL syntax is correct
- All referenced types exist
- No conflicts with existing types

The dry-run returns JSON schemas for the new types if compilation succeeds.

In [9]:
%run util.py
import json

config = util.load('registration_config')
spec_text = util.load('spec_text')
new_types = util.load('new_types')
kb_version = config['kb_version']

ws = KBWSUtils(kb_version=kb_version)

print(f"Dry-run registration of {config['module_name']} on {kb_version}")
print(f"New types: {new_types}")
print("=" * 80)

# Run dry-run (compile only, don't save)
result = ws.register_typespec_dryrun(spec_text, [], dryrun=True)

if isinstance(result, dict) and 'error' in result:
    print(f"\nDRY-RUN FAILED:")
    print(f"Error: {result['error']}")
else:
    print(f"\nDRY-RUN SUCCEEDED!")
    print(f"Returned {len(result)} type schemas")
    
    # Display the schemas for the new types
    for type_name in new_types:
        full_name = f"{config['module_name']}.{type_name}"
        if full_name in result:
            schema = json.loads(result[full_name])
            print(f"\n--- Schema for {full_name} ---")
            print(json.dumps(schema, indent=2)[:2000])
        else:
            # Check for versioned key
            matching = [k for k in result if type_name in k]
            if matching:
                for k in matching:
                    schema = json.loads(result[k])
                    print(f"\n--- Schema for {k} ---")
                    print(json.dumps(schema, indent=2)[:2000])

util.save('dryrun_result', result if not isinstance(result, dict) or 'error' not in result else {'error': str(result)})

2026-02-20 09:41:20,580 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-20 09:41:20,581 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-20 09:41:20,581 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-20 09:41:20,582 - __main__.NotebookUtil - INFO - Notebook name: RegisterGenomeDataLakeTables
2026-02-20 09:41:20,582 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-20 09:41:20,585 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-20 09:41:20,586 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-20 09:41:20,586 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token


Dry-run registration of KBaseFBA on appdev
New types: ['GenomeDataLakeTables']
Note: Registering new typespecs requires:
  1. Module ownership (request via request_module_ownership)
  2. Admin approval for new modules
  3. Valid KIDL syntax

DRY-RUN SUCCEEDED!
Returned 1 type schemas

--- Schema for KBaseFBA.GenomeDataLakeTables-3.0 ---
{
  "id": "GenomeDataLakeTables",
  "description": "GenomeDataLakeTables object: this object holds the DataLake tables with contextual data for the input genome set\n\n@metadata ws genomeset_ref as Genome set\n@metadata ws description as Description\n@metadata ws name as Name\n@metadata ws length(pangenome_data) as Number pangenomes",
  "type": "object",
  "original-type": "kidl-structure",
  "properties": {
    "description": {
      "type": "string",
      "original-type": "kidl-string"
    },
    "name": {
      "type": "string",
      "original-type": "kidl-string"
    },
    "genomeset_ref": {
      "type": "string",
      "original-type": "kidl-st

## Step 4: Register the Typespec (For Real)

**WARNING:** This cell will actually register the typespec with the workspace.
Only run after the dry-run succeeds.

This will:
1. Compile and save the typespec
2. Make the new types available (but not yet released)
3. The types will be usable in the target environment

In [10]:
%run util.py
import json

config = util.load('registration_config')
spec_text = util.load('spec_text')
new_types = util.load('new_types')
kb_version = config['kb_version']

# Safety check - set to True to actually register
CONFIRM_REGISTRATION = True  # Change to True to proceed

ws = KBWSUtils(kb_version=kb_version)

if CONFIRM_REGISTRATION:
    print(f"Registering typespec for {config['module_name']} on {kb_version}")
    print(f"New types: {new_types}")
    print("=" * 80)
    
    # Register for real (dryrun=False)
    result = ws.register_typespec_dryrun(spec_text, [], dryrun=False)
    
    if isinstance(result, dict) and 'error' in result:
        print(f"\nREGISTRATION FAILED:")
        print(f"Error: {result['error']}")
    else:
        print(f"\nREGISTRATION SUCCEEDED!")
        print(f"Returned {len(result)} type schemas")
        
        # Show registered types
        for type_name in new_types:
            matching = [k for k in result if type_name in k]
            for k in matching:
                print(f"  Registered: {k}")
    
    util.save('registration_result', result if not isinstance(result, dict) or 'error' not in result else {'error': str(result)})
else:
    print("Registration skipped. Set CONFIRM_REGISTRATION = True to proceed.")
    print(f"\nThis would register types {new_types} in module {config['module_name']} on {kb_version}")

2026-02-20 09:41:25,611 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-20 09:41:25,611 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-20 09:41:25,612 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-20 09:41:25,613 - __main__.NotebookUtil - INFO - Notebook name: RegisterGenomeDataLakeTables
2026-02-20 09:41:25,613 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-20 09:41:25,616 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-20 09:41:25,616 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-20 09:41:25,617 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token


Registering typespec for KBaseFBA on appdev
New types: ['GenomeDataLakeTables']
Note: Registering new typespecs requires:
  1. Module ownership (request via request_module_ownership)
  2. Admin approval for new modules
  3. Valid KIDL syntax

REGISTRATION SUCCEEDED!
Returned 1 type schemas
  Registered: KBaseFBA.GenomeDataLakeTables-3.0


## Step 5: Release the Module

**WARNING:** This releases the module, making the new types available for general use.
Only run after successful registration in Step 4.

After release, the `KBaseFBA.GenomeDataLakeTables` type will be available
for saving objects in any workspace.

In [11]:
%run util.py

config = util.load('registration_config')
kb_version = config['kb_version']
module_name = config['module_name']

# Safety check - set to True to release
CONFIRM_RELEASE = True  # Change to True to proceed

ws = KBWSUtils(kb_version=kb_version)

if CONFIRM_RELEASE:
    print(f"Releasing module: {module_name} on {kb_version}")
    print("=" * 80)
    
    result = ws.release_module(module_name)
    print(f"Result: {result}")
    
    if result.get('status') == 'success':
        print(f"\nReleased types: {result.get('released_types', [])}")
    else:
        print(f"\nRelease failed: {result.get('message', 'Unknown error')}")
    
    util.save('release_result', result)
else:
    print("Release skipped. Set CONFIRM_RELEASE = True to proceed.")
    print(f"\nThis would release module {module_name} on {kb_version}")

2026-02-20 09:41:34,271 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-20 09:41:34,271 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-20 09:41:34,271 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-20 09:41:34,272 - __main__.NotebookUtil - INFO - Notebook name: RegisterGenomeDataLakeTables
2026-02-20 09:41:34,272 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-20 09:41:34,275 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-20 09:41:34,275 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-20 09:41:34,276 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token


Releasing module: KBaseFBA on appdev
Result: {'status': 'success', 'released_types': ['KBaseFBA.FBAComparison-5.0', 'KBaseFBA.FBAPathwayAnalysis-1.0', 'KBaseFBA.EscherMap-4.0', 'KBaseFBA.FBA-14.1', 'KBaseFBA.regulatory_network-2.0', 'KBaseFBA.Classifier-1.1', 'KBaseFBA.GenomeDataLakeTables-3.0', 'KBaseFBA.BooleanGeneExpressionDataCollection-2.0', 'KBaseFBA.ModelTemplate-4.2', 'KBaseFBA.FBAModelSet-1.1', 'KBaseFBA.Gapfilling-7.0', 'KBaseFBA.ModelComparison-4.0', 'KBaseFBA.MissingRoleData-1.0', 'KBaseFBA.FBAPathwayAnalysisMultiple-1.0', 'KBaseFBA.SubsystemAnnotation-1.0', 'KBaseFBA.ClassifierResult-5.0', 'KBaseFBA.Gapgeneration-4.0', 'KBaseFBA.ReactionSensitivityAnalysis-5.1', 'KBaseFBA.FBAModel-14.0', 'KBaseFBA.NewModelTemplate-2.1', 'KBaseFBA.BooleanGeneExpressionData-1.0', 'KBaseFBA.ReactionProbabilities-2.0', 'KBaseFBA.ETC-6.0', 'KBaseFBA.PromConstraint-6.1', 'KBaseFBA.ClassifierTrainingSet-5.0']}

Released types: ['KBaseFBA.FBAComparison-5.0', 'KBaseFBA.FBAPathwayAnalysis-1.0', 'KBa

## Step 6: Verify Registration

Confirm the new type is available by fetching its type info from the workspace.
This verifies:
- The type exists in the registry
- The JSON schema matches expectations
- Metadata annotations are correct

In [11]:
%run util.py
import json

config = util.load('registration_config')
kb_version = config['kb_version']

ws = KBWSUtils(kb_version=kb_version)

type_name = f"{config['module_name']}.GenomeDataLakeTables"
print(f"Verifying type: {type_name} on {kb_version}")
print("=" * 80)

try:
    specs = ws.get_type_specs([type_name])
    
    if type_name in specs:
        spec_info = specs[type_name]
        print(f"\nType found: {type_name}")
        print(f"Type def: {spec_info.get('type_def', 'N/A')}")
        print(f"Description: {spec_info.get('description', 'N/A')}")
        print(f"Module versions: {spec_info.get('module_vers', [])}")
        print(f"Type versions: {spec_info.get('type_vers', [])}")
        
        # Show JSON schema
        json_schema = spec_info.get('json_schema', '')
        if json_schema:
            schema = json.loads(json_schema)
            print(f"\n--- JSON Schema ---")
            print(json.dumps(schema, indent=2))
        
        # Show spec def
        spec_def = spec_info.get('spec_def', '')
        if spec_def:
            print(f"\n--- KIDL Spec ---")
            print(spec_def)
        
        util.save('type_verification', spec_info)
        print("\nVerification PASSED!")
    else:
        print(f"\nType {type_name} not found in registry.")
        print("Available keys:", list(specs.keys()))
        
except Exception as e:
    print(f"\nVerification FAILED: {e}")
    print("The type may not be registered or released yet.")

2026-02-18 22:47:12,186 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-18 22:47:12,187 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-18 22:47:12,188 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-18 22:47:12,189 - __main__.NotebookUtil - INFO - Notebook name: RegisterGenomeDataLakeTables
2026-02-18 22:47:12,189 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-18 22:47:12,192 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-18 22:47:12,192 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-18 22:47:12,193 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-18 22:47:12,194 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Retrieving type specifications for 1 types


Verifying type: KBaseFBA.GenomeDataLakeTables on prod


2026-02-18 22:47:12,481 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Successfully retrieved 1 type specifications



Type found: KBaseFBA.GenomeDataLakeTables
Type def: KBaseFBA.GenomeDataLakeTables-1.0
Description: GenomeDataLakeTables object: this object holds the DataLake tables with contextual data for the input genome set

@metadata ws genomeset_ref as Genome set
@metadata ws description as Description
@metadata ws name as Name
@metadata ws length(pangenome_data) as Number pangenomes
Module versions: [1771476103441]
Type versions: ['KBaseFBA.GenomeDataLakeTables-0.1', 'KBaseFBA.GenomeDataLakeTables-1.0']

--- JSON Schema ---
{
  "id": "GenomeDataLakeTables",
  "description": "GenomeDataLakeTables object: this object holds the DataLake tables with contextual data for the input genome set\n\n@metadata ws genomeset_ref as Genome set\n@metadata ws description as Description\n@metadata ws name as Name\n@metadata ws length(pangenome_data) as Number pangenomes",
  "type": "object",
  "original-type": "kidl-structure",
  "properties": {
    "description": {
      "type": "string",
      "original-type"

## Step 7: Check Updated Module Info

After registration, verify the module now includes the new types
and that the module version was incremented.

In [12]:
%run util.py

config = util.load('registration_config')
kb_version = config['kb_version']
module_name = config['module_name']

ws = KBWSUtils(kb_version=kb_version)

print(f"Updated module info for: {module_name} on {kb_version}")
print("=" * 80)

module_info = ws.get_module_info(module_name)
if 'error' in module_info:
    print(f"Error: {module_info['error']}")
else:
    print(f"Owners: {module_info.get('owners', [])}")
    
    types = module_info.get('types', {})
    print(f"\nRegistered types ({len(types)}):")
    for type_name, type_ver in sorted(types.items()):
        # Highlight new types
        marker = " <-- NEW" if 'GenomeDataLakeTables' in type_name or 'PangenomeData' in type_name else ""
        print(f"  {type_name}: {type_ver}{marker}")

# Check versions
versions = ws.list_module_versions(module_name)
if 'error' not in versions:
    print(f"\nModule versions: {versions}")

util.save('updated_module_info', module_info)

2026-02-18 22:47:29,446 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-18 22:47:29,447 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-18 22:47:29,448 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token
2026-02-18 22:47:29,448 - __main__.NotebookUtil - INFO - Notebook name: RegisterGenomeDataLakeTables
2026-02-18 22:47:29,449 - __main__.NotebookUtil - INFO - Notebook environment detected
2026-02-18 22:47:29,451 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2026-02-18 22:47:29,452 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2026-02-18 22:47:29,453 - kbutillib.kb_ws_utils.KBWSUtils - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token


Updated module info for: KBaseFBA on prod
Owners: ['chenry']

Registered types (22):
  KBaseFBA.Classifier-1.1: {
  "id" : "Classifier",
  "description" : "",
  "type" : "object",
  "original-type" : "kidl-structure",
  "properties" : {
    "id" : {
      "type" : "string",
      "original-type" : "kidl-string"
    },
    "attribute_type" : {
      "type" : "string",
      "original-type" : "kidl-string"
    },
    "classifier_type" : {
      "type" : "string",
      "original-type" : "kidl-string"
    },
    "trainingset_ref" : {
      "type" : "string",
      "original-type" : "kidl-string",
      "id-reference" : {
        "attributes" : [ "KBaseFBA.ClassifierTrainingSet" ],
        "id-type" : "ws"
      }
    },
    "data" : {
      "type" : "string",
      "original-type" : "kidl-string"
    },
    "readable" : {
      "type" : "string",
      "original-type" : "kidl-string"
    },
    "correctly_classified_instances" : {
      "type" : "integer",
      "original-type" : "kidl-in

## Summary

| Step | Action | Safe to Run | Confirmation Required |
|------|--------|-------------|----------------------|
| 1 | Load and inspect KIDL spec | Yes | No |
| 2 | Check module info and ownership | Yes | No |
| 2b | Request module ownership | Requires confirmation | Yes |
| 3 | Dry-run registration (compile only) | Yes | No |
| 4 | Register typespec (for real) | Requires confirmation | Yes |
| 5 | Release module | Requires confirmation | Yes |
| 6 | Verify type registration | Yes | No |
| 7 | Check updated module info | Yes | No |

**Typical workflow:**
1. Run Steps 1-3 to validate the spec (safe, read-only)
2. Run Step 4 with `CONFIRM_REGISTRATION = True` to register
3. Run Step 5 with `CONFIRM_RELEASE = True` to release
4. Run Steps 6-7 to verify everything worked

**Type structure:**
```
GenomeDataLakeTables
  ├── description: string
  ├── name: string
  ├── genomeset_ref: KBaseCollections.GenomeSet reference
  └── pangenome_data: list<PangenomeData>
        ├── pangenome_id: string
        ├── pangenome_taxonomy: string
        ├── user_genomes: list<genome_ref>
        ├── datalake_genomes: list<string>
        └── sqllite_tables_handle_ref: handle reference
```